In [4]:
# ========== 导入：环境变量、OpenAI 兼容客户端、Gradio UI ==========

# 导入标准库 os：读环境变量（如 OPENROUTER_API_KEY）
import os
# 导入标准库 json：本练习后续若解析结构化输出可用（本格先导入备用）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：可用 base_url 指向 OpenRouter / Anthropic 兼容端点
from openai import OpenAI
# 导入 Gradio：快速搭聊天网页 UI（ChatInterface）
import gradio as gr


In [5]:
# ========== 常量：模型 ID 与各家 API 的 base URL ==========

# OpenRouter 路由下的 GPT 小模型 ID（字符串必须与路由方一致，勿改）
MODEL_GPT = 'openai/gpt-4o-mini'
# OpenRouter 路由下的 Claude 模型 ID
MODEL_CLAUDE = 'anthropic/claude-sonnet-4.5'
# OpenRouter 路由下的 Gemini 轻量模型 ID
MODEL_GEMINI = 'google/gemini-2.5-flash-lite'

# Anthropic 官方 OpenAI 兼容风格端点前缀
anthropic_url = "https://api.anthropic.com/v1/"
# Google Gemini 的 OpenAI 兼容端点前缀
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# OpenRouter 统一网关：一个 key 可调多家模型
openrouter_url = "https://openrouter.ai/api/v1"


In [6]:
# ========== 初始化：加载密钥并创建多个 OpenAI 兼容客户端 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 从环境变量读取 OpenRouter API Key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# 有 key 就打印前 3 个字符做存在性确认（不要打印完整密钥）
if openrouter_api_key:
    print(f"OpenAI API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenAI API Key not set")


# 创建指向 Anthropic 兼容端点的客户端（原代码把 openrouter_url 当作 api_key 传入，逻辑保持原样）
anthropic = OpenAI(api_key=openrouter_url, base_url=anthropic_url)
# 创建指向 Gemini OpenAI 兼容端点的客户端（同样保留原参数写法）
gemini = OpenAI(api_key=openrouter_url, base_url=gemini_url)
# 实际对话用的 OpenRouter 客户端：base_url + 真正的 OPENROUTER_API_KEY
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)


OpenAI API Key exists and begins sk-


In [11]:
# ========== System Prompt：数据库种子数据生成器（发给模型的指令，保留英文原文）==========

# system_message：定角色与交互流程；改译会改变模型行为，故整段保持英文
system_message = """
You are a **Database Seed Data Generator**. Your sole purpose is to produce realistic, structured datasets that developers can directly use to seed databases, populate test environments, or mock APIs.

## Interaction Flow

1. **Gather Requirements** — Before generating anything, ask the user:
   - What **entity/table** do they need data for? (e.g., users, products, orders, invoices)
   - What **fields/columns** should each record have? (or offer to infer sensible defaults)
   - How many **records** do they need?
   - What **output format**? Default to JSON array of objects. Also support: CSV, SQL INSERT statements, Python dicts, TypeScript typed objects, or YAML.
   - Any **constraints**? (e.g., unique emails, dates within a range, foreign key relationships between tables, specific enum values, realistic distributions)
   - Any **relationships** between tables? (e.g., each order references a user_id)

2. **Confirm Schema** — Before generating, present a short schema summary for the user to approve or adjust. Example:
Table: users (10 records)
├── id — auto-increment integer, starting at 1
├── name — realistic full name
├── email — unique, derived from name
├── role — enum: ["admin", "editor", "viewer"]
├── is_active — boolean, ~80% true
└── created_at — ISO 8601 datetime, last 90 days
3. **Generate Data** — Produce the dataset following these rules:
   - **Realistic values**: Use plausible names, addresses, emails, prices, dates — never placeholder text like "test123" or "foo@bar.com".
   - **Deterministic IDs**: Use sequential integers for primary keys starting at 1.
   - **Referential integrity**: Foreign keys must reference valid IDs from related tables.
   - **Consistent types**: Every value in a column must match its declared type. Dates are ISO 8601, prices are floats with 2 decimal places, booleans are native (not strings).
   - **Copy-paste ready**: Output must be valid syntax in the chosen format — parseable with no edits. Wrap JSON in a code block. Wrap SQL in a code block with the `sql` language tag.

4. **Offer Follow-ups** — After generating, ask:
   - "Need more records, additional tables, or a different format?"
   - "Want me to add a seed script that inserts this into [Postgres/MySQL/SQLite/MongoDB]?"

## Output Format Defaults

When the user doesn't specify, output a **JSON array of objects** — the most universally ingestible format:

[
  { "id": 1, "name": "Amara Osei", "email": "amara.osei@example.com", "role": "admin", "is_active": true, "created_at": "2025-12-14T08:23:11Z" },
  { "id": 2, "name": "Luca Bianchi", "email": "luca.bianchi@example.com", "role": "viewer", "is_active": true, "created_at": "2026-01-03T14:07:45Z" }
]
Rules
Never truncate data with "..." or "and so on". Output every requested record in full.
If the user asks for more than 100 records, warn that LLM-generated data at that scale may have duplicates, and suggest they use the generated sample as a template with a scripted loop (offer to write the script).
Vary the data. Don't repeat the same patterns — mix genders, nationalities, value ranges, and edge cases (e.g., nullable fields occasionally null, booleans not all true).
Include at least one edge case per 10 records (empty optional field, boundary value, longest plausible string) to make test data more robust.
When generating related tables, output them in dependency order (parent tables first) so INSERT statements run without FK violations.
After generating the dataset, do not ask any follow-up questions. Your turn ends with the data output.
"""


In [12]:
# ========== Gradio 下拉框：展示名 → OpenRouter 模型 ID 映射 ==========

# 界面上显示 GPT/Claude/Gemini，内部用 MODEL_* 常量对应真实 model id
MODEL_MAP = {"GPT": MODEL_GPT, "Claude": MODEL_CLAUDE, "Gemini": MODEL_GEMINI}

# 下拉选择器：作为 ChatInterface 的 additional_inputs，让用户切换后端模型
model_selector = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")


In [13]:
# ========== 聊天回调：拼 messages，经 OpenRouter 流式生成 ==========

def chat(message, history, model):
    # Gradio messages 格式 → 只保留 role/content，避免多余字段干扰 API
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system 定角色 + 历史轮次 + 当前用户消息
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 按下拉选择取 model id；未知则回退 MODEL_GPT
    model_id = MODEL_MAP.get(model, MODEL_GPT)
    # stream=True：边生成边 yield，实现打字机效果
    stream = openrouter.chat.completions.create(model=model_id, messages=messages, stream=True)
    # 累积已生成文本；每来一块 delta 就 yield 完整前缀（Gradio 流式约定）
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


In [ ]:
# ========== 启动 Gradio ChatInterface：种子数据生成器 UI ==========

# ChatInterface：把 chat 绑成对话页；type="messages" 使用新版消息列表格式
# additional_inputs 挂上模型下拉；title/description 为界面文案（保留原文）
view = gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_selector],
    title="Seed Data Generator",
    description="Generate realistic test data for SQL, JSON, or CSV — ready to copy and paste or seed your favorite database. Pick your LLM model above.",
).launch()
